# Notebook de ejemplos y validación
## Paquete `multasMadrid`

Este notebook demuestra el funcionamiento correcto de las clases `Cache`, `CacheURL` y `MadridFines`, incluyendo el manejo de excepciones.

In [ ]:
# Instalar el paquete si no está disponible
# !pip install -e ..

import sys
sys.path.insert(0, '..')  # Añadir el directorio raíz del proyecto al path

from multasMadrid import Cache, CacheError, CacheURL, MadridError, MadridFines, get_url

---
## 1. Clase `Cache`

### 1.1 Uso básico

In [ ]:
# Crear un objeto Cache para nuestra aplicación
cache = Cache(app_name="notebook_demo", obsolescence=1)

print(f"app_name:    {cache.app_name}")
print(f"cache_dir:   {cache.cache_dir}")
print(f"obsolescence: {cache.obsolescence} días")

In [ ]:
# Guardar datos en caché
cache.set("prueba1", "Hola, soy un dato guardado en caché")
cache.set("prueba2", "Otro dato de prueba")

print("¿Existe 'prueba1'?", cache.exists("prueba1"))
print("¿Existe 'no_existe'?", cache.exists("no_existe"))

In [ ]:
# Cargar datos
dato = cache.load("prueba1")
print("Dato cargado:", dato)

In [ ]:
# Antigüedad del fichero en milisegundos
edad_ms = cache.how_old("prueba1")
print(f"Antigüedad: {edad_ms:.2f} ms  ({edad_ms/1000:.4f} segundos)")

In [ ]:
# Borrar un elemento
cache.delete("prueba2")
print("¿Existe 'prueba2' tras delete()?", cache.exists("prueba2"))

# Borrar elemento que no existe: no lanza excepción
cache.delete("elemento_inexistente")
print("delete() de elemento inexistente: sin error")

In [ ]:
# Limpiar toda la caché
cache.set("a", "1")
cache.set("b", "2")
cache.clear()
print("¿Existe 'prueba1' tras clear()?", cache.exists("prueba1"))
print("¿Existe 'a' tras clear()?", cache.exists("a"))

### 1.2 Manejo de excepciones en `Cache`

In [ ]:
# load() con nombre inexistente → CacheError
try:
    cache.load("no_existe")
except CacheError as e:
    print(f"CacheError capturado correctamente: {e}")

In [ ]:
# how_old() con nombre inexistente → CacheError
try:
    cache.how_old("tampoco_existe")
except CacheError as e:
    print(f"CacheError capturado: {e}")

---
## 2. Clase `CacheURL`

### 2.1 Descarga con caché

In [ ]:
# Crear objeto CacheURL
cache_url = CacheURL(app_name="demo_urls", obsolescence=7)

# URL de prueba pública
url_prueba = "https://datos.madrid.es/egob/catalogo/210104-395-multas-circulacion-detalle.csv"

print("¿Está en caché?", cache_url.exists(url_prueba))

In [ ]:
# Primera llamada: descarga desde Internet
print("Descargando (primera vez)...")
contenido = cache_url.get(url_prueba)
print(f"Descargados {len(contenido)} caracteres")
print("¿Está ahora en caché?", cache_url.exists(url_prueba))

In [ ]:
# Segunda llamada: recupera de caché (NO hace petición HTTP)
print("Cargando desde caché (segunda vez)...")
contenido2 = cache_url.get(url_prueba)
print(f"Datos recuperados de caché: {len(contenido2)} caracteres")
print("¿Es el mismo contenido?", contenido == contenido2)

In [ ]:
# Antigüedad de la caché para esta URL
edad = cache_url.how_old(url_prueba)
print(f"Antigüedad de la caché: {edad:.2f} ms")

In [ ]:
# Eliminar caché de esa URL
cache_url.delete(url_prueba)
print("¿Existe tras delete()?", cache_url.exists(url_prueba))

---
## 3. Función `get_url`

Obtiene la URL del CSV de multas para un mes y año concretos mediante scraping.

In [ ]:
# Obtener URL de diciembre 2024
try:
    url_dic2024 = get_url(year=2024, month=12)
    print(f"URL encontrada para 12/2024: {url_dic2024}")
except MadridError as e:
    print(f"MadridError: {e}")

In [ ]:
# Mes/año inexistente → MadridError
try:
    url_inexistente = get_url(year=1990, month=1)
except MadridError as e:
    print(f"MadridError capturado correctamente: {e}")

---
## 4. Clase `MadridFines`

### 4.1 Carga de un mes

In [ ]:
# Crear objeto MadridFines
# obsolescence=30 → la caché es válida durante 30 días
mf = MadridFines(app_name="demo_multas", obsolescence=30)

print("Meses cargados inicialmente:", mf._loaded)
print("DataFrame vacío inicial:", mf._data.empty)

In [ ]:
# Cargar diciembre 2024
print("Cargando diciembre 2024 (puede tardar si no está en caché)...")
mf.add(2024, 12)

print(f"Meses cargados: {mf._loaded}")
print(f"Filas en el DataFrame: {len(mf._data)}")
mf._data.head(3)

In [ ]:
# Intentar cargar el mismo mes de nuevo: no duplica
filas_antes = len(mf._data)
mf.add(2024, 12)
filas_despues = len(mf._data)
print(f"Filas antes: {filas_antes} — Filas después de add() duplicado: {filas_despues}")
print("¿Se duplicaron los datos?", filas_antes != filas_despues)

### 4.2 Carga de múltiples meses

In [ ]:
# Cargar noviembre 2024 también
mf.add(2024, 11)
print(f"Meses cargados: {mf._loaded}")
print(f"Total filas: {len(mf._data)}")

### 4.3 Gráfico de multas por hora del día

In [ ]:
# Generar y guardar el gráfico
mf.fines_hour("evolucion_multas.png")
print("Gráfico guardado en: evolucion_multas.png")

# Mostrar el gráfico en el notebook
from IPython.display import Image
Image("evolucion_multas.png")

### 4.4 Distribución por calificación

In [ ]:
df_cal = mf.fines_calification()
print("Multas por calificación, mes y año:")
df_cal

### 4.5 Resumen de importes

In [ ]:
df_pagos = mf.total_payment()
print("Resumen de importes por mes/año:")
df_pagos

### 4.6 Manejo de excepciones en `MadridFines`

In [ ]:
# fines_hour() sin datos → MadridError
from multasMadrid import MadridFines, MadridError
mf_vacio = MadridFines(app_name="demo_vacio")

try:
    mf_vacio.fines_hour("test.png")
except MadridError as e:
    print(f"MadridError capturado: {e}")

In [ ]:
# fines_calification() sin datos → MadridError
try:
    mf_vacio.fines_calification()
except MadridError as e:
    print(f"MadridError capturado: {e}")

In [ ]:
# total_payment() sin datos → MadridError
try:
    mf_vacio.total_payment()
except MadridError as e:
    print(f"MadridError capturado: {e}")

In [ ]:
# add() con mes/año no disponible → MadridError
try:
    mf_vacio.add(1990, 1)
except MadridError as e:
    print(f"MadridError al intentar cargar datos de 01/1990: {e}")

---
## 5. Verificación de atributos privados

Los atributos `_data` y `_loaded` no deben poder modificarse directamente desde fuera de la clase.

In [ ]:
# Los atributos son 'privados' por convención (nombre con _)
# Solo se accede mediante la propia clase
print("Meses cargados (acceso de solo lectura):", mf._loaded)
print("Forma del DataFrame:", mf._data.shape)